Here we test out the sensitivity to backtests with respect to multiple parameters, eg. punishment with respect to CVaR, weights concentration, volatility and turnover.

In [ ]:
import numpy as np

def transaction_costs(trade_values, etf_spreads, portfolio_value):
    """
    trade_values: array of € amounts traded per ETF
    etf_spreads: half-spread % per ETF
    """
    # 1. Brokerage (IBKR tiered: 0.05% min €3.50)
    brokerage = np.maximum(3.50, 0.0005 * trade_values)
    
    # 2. Exchange + clearing (~€1.90 per trade)
    exchange = 1.90 * (trade_values > 0)
    
    # 3. Spread costs (round-trip = 2 * half-spread)
    spreads = 2 * etf_spreads * trade_values
    
    total_costs = brokerage + exchange + spreads
    return np.sum(total_costs) / portfolio_value  # % drag

# ETFs spreads (bps) - estimated
etf_spreads = np.array([0.0075, 0.050, 0.100, 0.035, 0.130, 0.030, 0.030]) / 100

# # In backtest:
# prev_weights = np.ones(8) / 8  # initial equal weight
# total_tc_costs = []

# for w_opt in weights_opt:
#     tc_cost, turnover = transaction_costs(prev_weights, w_opt, 100000, etf_spreads)
#     total_tc_costs.append(tc_cost)
#     returns_with_costs = returns - tc_cost  # subtract from period return
#     prev_weights = w_opt

def realistic_rebalance(portfolio_value, target_weights, prev_weights, etf_prices, prev_shares, weight_diff_rebalance):
    """
    prev_shares: (N,) current shares held (persistent across calls)
    Returns: new_shares, trade_values, tc_drag, actual_weights
    """

    diff = np.abs(prev_weights - target_weights)
    
    # Target values excluding cash
    target_etf_values = portfolio_value * target_weights[:-1]
    
    # Target shares (not rounded)
    target_shares = target_etf_values / etf_prices
    
    # Trades required
    shares_traded = np.abs(target_shares - prev_shares)
    trade_values = shares_traded * etf_prices

    if diff[diff >= weight_diff_rebalance].any():
        return target_shares, trade_values, 0.0, target_weights, portfolio_value
    
    # Transaction cost drag
    tc_drag = transaction_costs(trade_values, etf_spreads, portfolio_value)
    
    # Actual post-trade portfolio value (after costs)
    new_portfolio_value = portfolio_value - tc_drag * portfolio_value
    
    # Actual weights (including rounding error)
    actual_etf_values = target_shares * etf_prices
    cash_value = new_portfolio_value - np.sum(actual_etf_values)
    actual_weights = np.append(actual_etf_values / new_portfolio_value, cash_value / new_portfolio_value)
    actual_weights = np.array(actual_weights)
    
    return target_shares, trade_values, tc_drag, actual_weights, new_portfolio_value

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]
date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
volumes_all = np.empty((T, N))
prices_all = np.empty((T, N))
dfs_new = []
for i, df in enumerate(dfs):
    df = df[::-1]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")  # Handles str/NaN/object
    returns = df["Price"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)
    volumes_all[:, i] = df["Vol."].apply(correct_volume).values[1:]
    prices_all[:,i] = df["Price"].values[1:]
    dfs_new.append(df)

all_dates = dfs[0]["Date"].values[1:]
volumes_all = np.nan_to_num(volumes_all, nan=0.0)

In [ ]:
import tensorflow as tf

def max_drawdown(prices, weights):
    # prices (T,N)
    # weights (N,)
    T, N = prices.shape
    portfolio_values = prices @ weights # (T,)
    highest_peak = -1
    worst_pct = 0.0
    for t in range(T):
        highest_peak = max(highest_peak, portfolio_values[t])
        worst_pct = min(worst_pct, portfolio_values[t] / highest_peak - 1)
    return worst_pct

def perform_validation(w, validation_data, validation_prices):
    # validation data is shape (T, N)

    cash_returns = np.zeros((validation_data.shape[0], 1))  # or use risk-free rate
    validation_data_with_cash = np.hstack([validation_data, cash_returns])
    #print('shape of validation_data_with_cash:', validation_data_with_cash.shape)
    #print('weights shape:', w.shape)

    # Reshape w to (8, 1) for matrix multiplication
    w = tf.reshape(w, (-1, 1))

    # perform metrics
    total_return = validation_data_with_cash @ w  # shape (T, 1)
    total_return = total_return.numpy().flatten()  # convert to (T,)

    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return < 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)

    md = max_drawdown(validation_prices, w[:-1].numpy())

    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
    

In [ ]:
def lower_part_mean(matrix):
    n, _ = matrix.shape
    s = 0.0
    for i in range(1,n):
        for j in range(i):
            s += matrix[i,j]
    N = (n-1)*n/2
    return s / N

def compute_statistics_rolling(returns, window_size, stepsize):
    print('returns:'); print(returns)
    all_corrs = []
    all_volas = []
    T, N = returns.shape
    w = np.ones(N) / N
    #print('T:', T)
    #print('testetst')
    for idx in range(0, T - window_size, stepsize):
        section = returns[idx:idx+window_size,:]
        #print('section shape:', section.shape)
        corr = np.corrcoef(section, rowvar=False) # (N,N)
        #print('appending', lower_part_mean(corr))
        all_corrs.append(lower_part_mean(corr))
        returns_total = section @ w
        #print('returns_total:', returns_total)
        std = np.std(returns_total)
        all_volas.append(std)
    
    all_corrs = np.array(all_corrs)
    all_volas = np.array(all_volas)
    #print('result:'); print(all_corrs)
    return all_corrs, all_volas


In [ ]:
# run

corrs, volas = compute_statistics_rolling(returns_all, 90, 30)

print('mean corrs:', np.mean(corrs))
print('mean volas:', np.mean(volas))
print('highest corrs:', np.max(corrs))
print('lowest corrs:', np.min(corrs))
print('lowest volas:', np.min(volas))
print('highest volas:', np.max(volas))
# results:
# based on this, determine threshold for correlation and volatility

vola_thresh = np.percentile(volas, 75)
corr_thresh = np.percentile(corrs, 75)

corrs_mean = np.mean(corrs)
corrs_std = np.std(corrs)
volas_mean = np.mean(volas)
volas_std = np.std(volas)

In [ ]:
import tensorflow as tf

learning_rate = 0.01
alpha = 0.05   # tail probability for CVaR
num_steps = 2000
min_weight = 1/9 # 1/N = 1/7, so a little less
max_weight = 0.3
a = 1.0 # for stress environment tuning
b = 1.0 # for stress environment tuning
weight_diff_rebalance = 0.10 # minimum weight difference for a portfolio rebalance
portfolio_value = 10_000

def sigmoid(x):
    return 1/(1+np.exp(-x))

def normalize(x):
    return (x - tf.reduce_mean(x)) / (tf.math.reduce_std(x) + 1e-8)

def rank_transform(x):
    ranks = tf.argsort(tf.argsort(x))
    return tf.cast(ranks, tf.float32) / tf.cast(tf.size(x), tf.float32)

def gradient(train_data, vol_data, params):

    lambda_, gamma, tau = params

    # returns (weights, mode, had_to_cap_weights, stress_factor, turnover_values)
    # where mode = 0 if returned normal (optimal) weights,
    # 1 = if returned equal because of correlations
    # 2 = if returned equal because of volatility

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    turnover_values = []

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    V_tf = tf.convert_to_tensor(vol_data, dtype=tf.float32) # (T,N)
    #cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    corr = np.corrcoef(train_data, rowvar=False)
    mean_corr = lower_part_mean(corr)
    #print('mean corr:', mean_corr)
    corr_standardized = (mean_corr - corrs_mean) / corrs_std

    w_prev = tf.identity(w)

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            # momentum per asset (N,)
            momentum = tf.reduce_mean(X_tf[-60:], axis=0)

            # volume per asset (N,)
            vol_short = tf.reduce_mean(V_tf[-10:], axis=0)
            vol_long  = tf.reduce_mean(V_tf[-60:], axis=0)
            volume_signal = (vol_short - vol_long) / (vol_long + 1e-8)

            volatility = tf.math.reduce_std(X_tf[-60:], axis=0)
            risk_adj_momentum = momentum / (volatility + 1e-8)

            m1 = normalize(momentum)
            m2 = normalize(risk_adj_momentum)
            m3 = normalize(volume_signal)
            #print('m1, m2, m3:', m1, m2, m3)

            # combine (N,)
            mu = 0.5 * m1 + 0.3 * m2 + 0.2 * m3
            mu = rank_transform(mu)

            turnover_penalty = tau * tf.reduce_sum(tf.square(w - w_prev))
            turnover = tf.reduce_sum(
                tf.abs(w_normalized - w_prev)
            ).numpy()
            turnover_values.append(turnover)

            expected_return = tf.tensordot(w_normalized, mu, axes=1)
            weights_penalty = gamma * tf.reduce_sum(tf.math.square(w_col))

            market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
            market_vol = tf.math.reduce_std(X_tf[-20:])

            objective = -(expected_return - lambda_ * cvar) + weights_penalty + turnover_penalty

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        w_prev = tf.identity(w)

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = train_data @ optimal_w # (T,)
    volatility = np.std(combined_returns)
    volatility_standardized = (volatility - volas_mean) / volas_std

    stress_score = sigmoid(a * corr_standardized + b * volatility_standardized)
    chosen_w = stress_score * np.ones(N) / N + (1 - stress_score) * optimal_w

    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    cash_weight = crash_signal * 0.5  # up to 50% cash
    w_risky = (1 - cash_weight) * chosen_w
    w_final = tf.concat([w_risky, [cash_weight]], axis=0)

    if tf.reduce_any(w_final <= min_weight):
        w_final = np.clip(w_final, a_min = min_weight, a_max = max_weight)
        return w_final / np.sum(w_final), True, stress_score, np.mean(turnover_values)
    return w_final.numpy(), False, stress_score, np.mean(turnover_values)

def rolling_window(params, window_size, validation_size, stepsize, returns_data, volume_data, price_data, portfolio_init=10_000):
    # data is (T,N)

    # lambda_, gamma, tau = params

    sharpes_w_opt= []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_eqw = []

    mds_w_opt = []
    mds_w_eqw = []

    stress_numbers = []
    turnover_all = []

    T, N = returns_data.shape
    weights_opt = []

    clip_counter = 0
    total_counter = 0

    portfolio_values = [portfolio_init]
    prev_shares = np.zeros(N)  # Start with no positions
    tc_drags = [] # drag trading costs (pct)
    w_prev = np.ones(N+1)/(N+1)  # Start with equal weights

    for idx in range(0, len(returns) - window_size - validation_size, stepsize):
        train_data_returns = returns_data[idx:idx+window_size]
        train_data_volume = volume_data[idx:idx+window_size]
        train_data_prices = price_data[idx:idx+window_size]

        val_data_returns = returns_data[idx+window_size:idx+window_size+validation_size]
        val_data_prices = price_data[idx+window_size:idx+window_size+validation_size]

        w, applied_clip, stress, turnovers = gradient(train_data_returns, train_data_volume, params)
        weights_opt.append(w)
        if applied_clip:
            clip_counter += 1
        total_counter += 1
        turnover_all.append(turnovers)
        stress_numbers.append(stress)

        # Prices at rebalance (last train day)
        rebalance_prices = train_data_prices[-1]
        current_value = portfolio_values[-1]
        
        # Execute rebalance
        new_shares, trade_vals, tc_drag, actual_weights, new_value = realistic_rebalance(
            current_value, w, w_prev, rebalance_prices, prev_shares, weight_diff_rebalance
        ) # trade vals = dollar values of etfs, new_value = new portfolio balance

        prev_shares = new_shares[:]
        portfolio_values.append(new_value)
        tc_drags.append(tc_drag)

        ( # replaced w with actual_weights
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(actual_weights, val_data_returns, val_data_prices)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)
        mds_w_opt.append(md)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(np.ones(N+1) / (N+1), val_data_returns, val_data_prices)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)
        mds_w_eqw.append(md)

        w_prev = actual_weights.copy()

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_opt)],
        "Highest maximum drawdowns": [np.min(mds_w_eqw), np.min(mds_w_opt)],
        "Mean maximum drawdowns": [np.mean(mds_w_eqw), np.mean(mds_w_opt)]
    }, index=["Equal weights", "Return + CVaR"])

    weights_opt = np.array(weights_opt)

    return df, np.mean(weights_opt, axis=0), weights_opt, clip_counter / total_counter, stress_numbers, turnover_all, tc_drags

Backtest with varying parameter values

In [ ]:
# these are now a range of values

lambdas = [0.3, 0.5, 0.8]  # CVaR penalty
gammas = [0.03, 0.05, 0.08]
taus = [0.25, 0.5, 0.75] # to start

clip_pcts = np.empty((3,3,3))
costs = np.empty((3,3,3))
stresses = np.empty((3,3,3))
turnovers = np.empty((3,3,3))
weights_all = np.empty((3,3,3), dtype=object)
sharpes = np.empty((3,3,3))
sortinos = np.empty((3,3,3))
cvars = np.empty((3,3,3))
vars = np.empty((3,3,3))
meanreturns = np.empty((3,3,3))
highestdrawdowns = np.empty((3,3,3))

for i,lambda_ in enumerate(lambdas):
    for j,gamma in enumerate(gammas):
        for k,tau in enumerate(taus):

            df, weights_opt_mean, weights_opt, clip_pct, stress, turnover, cost = rolling_window(
                params=[lambda_, gamma, tau],
                window_size=252*2,
                validation_size=252,
                stepsize=252,
                returns_data=returns_all,
                volume_data=volumes_all,
                price_data=prices_all
            )
            clip_pcts[i,j,k] = clip_pct
            costs[i,j,k] = np.mean(cost)
            stresses[i,j,k] = np.mean(stress)
            turnovers[i,j,k] = np.mean(turnover)
            sortinos[i,j,k] = df.iloc["Return + CVaR", "Sortinos"]
            sharpes[i,j,k] = df.iloc["Return + CVaR", "Sharpes"]
            cvars[i,j,k] = df.iloc["Return + CVaR", "Cvars"]
            vars[i,j,k] = df.iloc["Return + CVaR", "Vars"]
            meanreturns[i,j,k] = df.iloc["Return + CVaR", "MeanReturn"]
            highestdrawdowns[i,j,k] = df.iloc["Return + CVaR", "Highest maximum drawdowns"]
            weights_all[i,j,k] = weights_opt_mean

In [ ]:
from plotly.subplots import make_subplots

sharpes_img = make_subplots(rows=1, cols=3)
sharpes_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=sharpes[:,:,0]), row=1, col=1)
sharpes_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=sharpes[:,:,1]), row=1, col=2)
sharpes_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=sharpes[:,:,2]), row=1, col=3)
sharpes_img.update_layout(title="Sharpes - X=gammas, Y=lambdas")
sharpes_img.show()


In [ ]:
sortinos_img = make_subplots(rows=1, cols=3)
sortinos_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=sortinos[:,:,0]), row=1, col=1)
sortinos_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=sortinos[:,:,1]), row=1, col=2)
sortinos_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=sortinos[:,:,2]), row=1, col=3)
sortinos_img.update_layout(title="Sortinos - X=gammas, Y=lambdas")
sortinos_img.show()


In [ ]:
cvars_img = make_subplots(rows=1, cols=3)
cvars_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=cvars[:,:,0]), row=1, col=1)
cvars_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=cvars[:,:,1]), row=1, col=2)
cvars_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=cvars[:,:,2]), row=1, col=3)
cvars_img.update_layout(title="CVaRs - X=gammas, Y=lambdas")
cvars_img.show()


In [ ]:
vars_img = make_subplots(rows=1, cols=3)
vars_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=vars[:,:,0]), row=1, col=1)
vars_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=vars[:,:,1]), row=1, col=2)
vars_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=vars[:,:,2]), row=1, col=3)
vars_img.update_layout(title="VaRs - X=gammas, Y=lambdas")
vars_img.show()


In [ ]:
highestdrawdowns_img = make_subplots(rows=1, cols=3)
highestdrawdowns_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=highestdrawdowns_img[:,:,0]), row=1, col=1)
highestdrawdowns_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=highestdrawdowns_img[:,:,1]), row=1, col=2)
highestdrawdowns_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=highestdrawdowns_img[:,:,2]), row=1, col=3)
highestdrawdowns_img.update_layout(title="Highest Drawdowns - X=gammas, Y=lambdas")
highestdrawdowns_img.show()


In [ ]:
meanreturns_img = make_subplots(rows=1, cols=3)
meanreturns_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=meanreturns_img[:,:,0]), row=1, col=1)
meanreturns_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=meanreturns_img[:,:,1]), row=1, col=2)
meanreturns_img.add_trace(go.Heatmap(x=gammas, y=lambdas, z=meanreturns_img[:,:,2]), row=1, col=3)
meanreturns_img.update_layout(title="Mean Returns - X=gammas, Y=lambdas")
meanreturns_img.show()
